# state_objective_fold_native_cluster_targets_v1

Goal: close the previous blocker where fold-local state-objective targets had zero rebound cluster rows. This notebook discovers rebound clusters only from each fold's training dates, rebuilds fold-local targets, and rechecks the BC label surface.

External basis:
- Qlib: keep data/model/strategy/backtest workflow surfaces decoupled and replayable. <https://github.com/microsoft/qlib>
- FinRL: reproducible DRL trading pipelines should keep data processing, environment, agent, and validation stages explicit. <https://github.com/AI4Finance-Foundation/FinRL>
- Backtest-overfitting/PBO literature: model selection on repeated backtests can overfit holdout evidence, so fold-native targets and walk-forward gates remain required before promotion. <https://doi.org/10.1090/conm/729/14680>

This is a diagnostic target-generation run, not a promotion run.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.config import load_profile
from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveReboundClusterConfig,
    build_fold_local_state_objective_training_targets,
    discover_rebound_clusters_from_daily_path,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
)
from src.soxl_quant.policy.policy_builder import build_state_objective_warm_start_dataset_from_frame

EXPERIMENT = "state_objective_fold_native_cluster_targets_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
TEACHER_CARD = ROOT / "runtime" / "reward" / "card" / "state_conditioned_rebound_risk_router_v1"
PREVIOUS_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_local_target_generation_v1"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
CARD.mkdir(parents=True, exist_ok=True)

candidate = "state_conditioned_rebound_risk_router_v1"
daily_path = pd.read_csv(TEACHER_CARD / "selected_strict_oos_daily.csv", parse_dates=["datetime"])
feature_frame = pd.read_pickle(FRAME_PATH)

print({
    "experiment": EXPERIMENT,
    "daily_rows": int(len(daily_path)),
    "daily_start": str(daily_path["datetime"].min().date()),
    "daily_end": str(daily_path["datetime"].max().date()),
    "feature_rows": int(len(feature_frame)),
})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'state_objective_fold_native_cluster_targets_v1', 'daily_rows': 1569, 'daily_start': '2020-01-02', 'daily_end': '2026-03-31', 'feature_rows': 2973}


## Fold definitions

These are the same training/evaluation splits used by the preceding fold-local target audit. The new piece is that cluster discovery happens inside each `train_start` to `train_end` interval.

In [2]:
folds = [
    {
        "name": "wf_2022_train",
        "source_window": "selected_strict_oos",
        "train_start": "2020-01-02",
        "train_end": "2021-12-31",
        "eval_start": "2022-01-03",
        "eval_end": "2022-12-30",
    },
    {
        "name": "wf_2023_2024_train",
        "source_window": "selected_strict_oos",
        "train_start": "2020-01-02",
        "train_end": "2022-12-30",
        "eval_start": "2023-01-03",
        "eval_end": "2024-12-31",
    },
]

pd.DataFrame(folds)

,name,source_window,train_start,train_end,eval_start,eval_end
0,wf_2022_train,selected_strict_oos,2020-01-02,2021-12-31,2022-01-03,2022-12-30
1,wf_2023_2024_train,selected_strict_oos,2020-01-02,2022-12-30,2023-01-03,2024-12-31


## Fold-native rebound cluster discovery

The previous audit used cluster metrics selected outside these folds. Here the cluster table is regenerated from the train-window daily path itself. `min_rows=3` is used to avoid treating isolated one-day rebound flags as training targets.

In [3]:
cluster_config = StateObjectiveReboundClusterConfig(min_rows=3)
cluster_frames = []
for fold in folds:
    cluster_frames.append(
        discover_rebound_clusters_from_daily_path(
            daily_path,
            candidate=candidate,
            fold=fold["name"],
            train_start=fold["train_start"],
            train_end=fold["train_end"],
            config=cluster_config,
        )
    )

fold_native_clusters = (
    pd.concat(cluster_frames, ignore_index=True)
    if cluster_frames
    else pd.DataFrame()
)
fold_native_clusters.to_csv(CARD / "fold_native_rebound_clusters.csv", index=False)
fold_native_clusters

,candidate,fold,cluster,start,end,rows,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,state_conditioned_rebound_risk_router_v1,wf_2022_train,wf_2022_train_rebound_1,2020-06-02,2020-06-08,5,8.222520,19.030862,-10.808342,0.368728,0.512498
1,state_conditioned_rebound_risk_router_v1,wf_2022_train,wf_2022_train_rebound_2,2020-11-24,2020-11-30,4,3.970737,6.183141,-2.212404,0.519334,0.612748
2,state_conditioned_rebound_risk_router_v1,wf_2023_2024_train,wf_2023_2024_train_rebound_1,2020-06-02,2020-06-08,5,8.222520,19.030862,-10.808342,0.368728,0.512498
3,state_conditioned_rebound_risk_router_v1,wf_2023_2024_train,wf_2023_2024_train_rebound_2,2020-11-24,2020-11-30,4,3.970737,6.183141,-2.212404,0.519334,0.612748
4,state_conditioned_rebound_risk_router_v1,wf_2023_2024_train,wf_2023_2024_train_rebound_3,2022-07-29,2022-08-08,7,0.809383,3.580439,-2.771056,1.447991,0.206288


## Rebuild fold-local targets with fold-native clusters

In [4]:
daily_paths = {"selected_strict_oos": daily_path}
fold_targets = build_fold_local_state_objective_training_targets(
    daily_paths,
    folds=folds,
    cluster_metrics=fold_native_clusters,
    candidate=candidate,
)
fold_targets.to_csv(CARD / "fold_native_cluster_targets.csv", index=False)

fold_target_summary = (
    fold_targets.groupby("fold", dropna=False)
    .agg(
        rows=("datetime", "count"),
        unique_dates=("datetime", "nunique"),
        rebound_rows=("is_rebound_cluster", "sum"),
        downtrend_rows=("is_downtrend_state", "sum"),
        max_objective_weight=("objective_weight", "max"),
        mean_teacher_position=("teacher_position", "mean"),
        mean_rebound_target_position=("rebound_target_position", "mean"),
        mean_risk_target_position=("risk_target_position", "mean"),
    )
    .reset_index()
)
fold_target_summary.to_csv(CARD / "fold_target_summary.csv", index=False)
fold_target_summary

,fold,rows,unique_dates,rebound_rows,downtrend_rows,max_objective_weight,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position
0,wf_2022_train,505,505,9,234,226.658515,0.207103,0.207103,0.193649
1,wf_2023_2024_train,756,756,16,478,15.371156,0.161971,0.163302,0.152684


## Compare action-label health

This uses the same feature frame and candidate profile set as the previous audit. Passing this screen only means the target/action surface is not degenerate; it does not remove the teacher-source provenance blocker.

In [5]:
profiles = ["risk_budget_usage_v1", "balanced", "balanced_rule_residual_v22"]
label_rows = []
fold_label_rows = []
for profile_name in profiles:
    profile = load_profile(profile_name)
    obs, actions = build_state_objective_warm_start_dataset_from_frame(
        feature_frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    dist = pd.Series(actions).value_counts(normalize=True).sort_index()
    label_rows.append(
        {
            "profile": profile_name,
            "samples": int(len(actions)),
            "obs_dim": int(obs.shape[1]),
            "unique_actions": int(dist.shape[0]),
            "dominant_action": int(dist.idxmax()),
            "dominant_action_share": float(dist.max()),
            "action_entropy": float(-(dist * np.log(dist + 1e-12)).sum()),
        }
    )
    for fold_name, subset in fold_targets.groupby("fold"):
        fold_obs, fold_actions = build_state_objective_warm_start_dataset_from_frame(
            feature_frame,
            targets=subset,
            profile=profile,
            sample_size=0,
        )
        fold_dist = pd.Series(fold_actions).value_counts(normalize=True).sort_index()
        fold_label_rows.append(
            {
                "fold": fold_name,
                "profile": profile_name,
                "samples": int(len(fold_actions)),
                "obs_dim": int(fold_obs.shape[1]),
                "unique_actions": int(fold_dist.shape[0]),
                "dominant_action": int(fold_dist.idxmax()),
                "dominant_action_share": float(fold_dist.max()),
                "action_entropy": float(-(fold_dist * np.log(fold_dist + 1e-12)).sum()),
            }
        )

label_comparison = pd.DataFrame(label_rows).sort_values(
    ["dominant_action_share", "action_entropy"], ascending=[True, False]
).reset_index(drop=True)
fold_label_comparison = pd.DataFrame(fold_label_rows).sort_values(
    ["fold", "dominant_action_share", "profile"]
).reset_index(drop=True)
label_comparison.to_csv(CARD / "profile_label_comparison.csv", index=False)
fold_label_comparison.to_csv(CARD / "fold_profile_label_comparison.csv", index=False)

readiness_scores = score_state_objective_bc_readiness(label_comparison, target_scope="fold_local")
readiness_summary = summarize_state_objective_bc_readiness(readiness_scores)
readiness_scores.to_csv(CARD / "bc_readiness_scores.csv", index=False)
label_comparison

,profile,samples,obs_dim,unique_actions,dominant_action,dominant_action_share,action_entropy
0,risk_budget_usage_v1,755,315,5,1,0.396026,1.202464
1,balanced,755,79,9,0,0.903311,0.457479
2,balanced_rule_residual_v22,755,340,5,0,0.964238,0.197395


In [6]:
readiness_summary

{'decision': 'state_objective_bc_ready_for_training',
 'selected_profile': 'risk_budget_usage_v1',
 'selected_target_scope': 'fold_local',
 'selected_bc_readiness_gate': True,
 'selected_label_distribution_gate': True,
 'selected_fold_local_scope_gate': True,
 'selected_dominant_action_share': 0.39602649006622515,
 'selected_unique_actions': 5,
 'selected_samples': 755,
 'selected_bc_readiness_loss': 0.0,
 'next_mechanism': 'run_fold_local_bc_warm_start_then_walkforward_ppo'}

## Selection and conclusion

The rebound-target coverage blocker is closed for this audit path if `rebound_rows > 0`. Training remains blocked because the teacher daily path is still the selected global diagnostic router path; the next valid training run needs fold-native teacher path generation or replay before BC/PPO.

In [7]:
previous_summary_path = PREVIOUS_CARD / "summary.json"
previous_summary = json.loads(previous_summary_path.read_text()) if previous_summary_path.exists() else {}
previous_rebound_rows = int(
    sum(row.get("rebound_rows", 0) for row in previous_summary.get("fold_target_summary", []))
)
current_rebound_rows = int(fold_target_summary["rebound_rows"].sum())
cluster_gap_closed = current_rebound_rows > previous_rebound_rows and current_rebound_rows > 0

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "fold_native_cluster_targets_generated_no_training",
    "promotion_allowed": False,
    "training_allowed": False,
    "selected_profile_for_next_training": readiness_summary["selected_profile"],
    "readiness_decision_if_source_is_fold_native": readiness_summary["decision"],
    "cluster_gap_closed": bool(cluster_gap_closed),
    "previous_rebound_rows": previous_rebound_rows,
    "current_rebound_rows": current_rebound_rows,
    "cluster_source": "fold_native_train_window_rebound_state",
    "source_provenance_blocker": (
        "teacher daily path is still selected by the global diagnostic router screen; "
        "regenerate or replay teacher paths inside each train fold before using these labels for training"
    ),
    "next_mechanism": "regenerate_fold_native_teacher_paths_then_budget_usage_bc_warm_start_ppo",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "fold_native_teacher_source_required": True,
        "label_distribution_gate_required": True,
    },
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://doi.org/10.1090/conm/729/14680",
    ],
    "sidecars": [
        "fold_native_rebound_clusters.csv",
        "fold_native_cluster_targets.csv",
        "fold_target_summary.csv",
        "profile_label_comparison.csv",
        "fold_profile_label_comparison.csv",
        "bc_readiness_scores.csv",
        "summary.json",
    ],
}
summary = {
    "selection": selection,
    "fold_native_clusters": fold_native_clusters.to_dict(orient="records"),
    "fold_target_summary": fold_target_summary.to_dict(orient="records"),
    "label_comparison": label_comparison.to_dict(orient="records"),
    "fold_label_comparison": fold_label_comparison.to_dict(orient="records"),
    "readiness_summary": readiness_summary,
    "conclusion": [
        f"Fold-native discovery raised rebound target rows from {previous_rebound_rows} to {current_rebound_rows}.",
        f"{readiness_summary['selected_profile']} remains the healthiest BC action surface under fold-local targets.",
        "No training or promotion is allowed yet because the teacher path source is still globally selected rather than fold-native.",
    ],
}

(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True, default=str))
selection

{'experiment': 'state_objective_fold_native_cluster_targets_v1',
 'created_at': '2026-05-25T11:10:17.401728+00:00',
 'decision': 'fold_native_cluster_targets_generated_no_training',
 'promotion_allowed': False,
 'training_allowed': False,
 'selected_profile_for_next_training': 'risk_budget_usage_v1',
 'readiness_decision_if_source_is_fold_native': 'state_objective_bc_ready_for_training',
 'cluster_gap_closed': True,
 'previous_rebound_rows': 0,
 'current_rebound_rows': 25,
 'cluster_source': 'fold_native_train_window_rebound_state',
 'source_provenance_blocker': 'teacher daily path is still selected by the global diagnostic router screen; regenerate or replay teacher paths inside each train fold before using these labels for training',
 'next_mechanism': 'regenerate_fold_native_teacher_paths_then_budget_usage_bc_warm_start_ppo',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd_max_pct': 15.0,
  'walkforward_required': True,
  'fold_native_teacher_source_requi